<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec11_motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec11_motion.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第11回 動き検出処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec11_motion.m` を Python に移植したもの。第11回のスライド（vie2026-11）で使う図と数値を計算する。エッジ画像のスペクトル，大域的定速移動モデルのスペクトル（速度に応じて傾く），ブロックマッチング法（差の絶対値和 SAD）による動き推定，位相限定相関（POC）を扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

教科書には動き推定の章がないため，記号は教科書の関連箇所に合わせる。
- 連続変数の角周波数を $\boldsymbol{\nu}=(\nu_\mathrm{v}\ \nu_\mathrm{h}\ \nu_\mathrm{t})^\top$ とする（$\nu_\mathrm{t}$ は時間方向）。
- 動きベクトルは $\boldsymbol{m}=[m_\mathrm{v}\ \ m_\mathrm{h}]^\top$，すなわち**（垂直，水平）の順**に並べる（教科書 3.1 節の $\boldsymbol{m}$ と同じ順）。
- 差の絶対値和は教科書 7.3 節 COLUMN「予測符号化」の $\mathrm{SAD}(\boldsymbol{m};b)=\sum_{\boldsymbol{n}\in\mathcal{N}_b}|x_{n_\mathrm{t}}[\boldsymbol{n}]-x_{m_\mathrm{t}}[\boldsymbol{n}+\boldsymbol{m}]|$ に従う。$\boldsymbol{m}$ は現フレームのブロックの位置から，参照フレーム内で一致するブロックの位置へのずれである。
- 相互相関とクロススペクトルは教科書 4.2.3 項，位相限定相関は同項の COLUMN「位相限定相関」に従う。

画像処理には scikit-image，SciPy，NumPy を使う。前回スライドの例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy import signal, stats
from skimage.filters import gaussian
from skimage.transform import resize
from skimage.util import img_as_float
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

## 標本化によるスペクトルの周期化（前回スライドの図を踏襲）
斜めに傾いた細長いスペクトル $\tilde{u}(\boldsymbol{\nu})$（円形とは限らない）を，標本化行列 $\boldsymbol{L}=\mathrm{diag}(\Delta_\mathrm{v},\Delta_\mathrm{h})$ の直交標本化で周期化する（教科書 4.1.3 項「FT と DSFT の関係」）。

$$\tilde{x}(\boldsymbol{\nu})=\frac{1}{|\det(\boldsymbol{L})|}\sum_{\boldsymbol{m}\in\mathbb{Z}^2}\tilde{u}(\boldsymbol{\nu}-2\pi\boldsymbol{L}^{-\top}\boldsymbol{m})$$

図では $\Delta_\mathrm{v}=\Delta_\mathrm{h}=1$ とし，周期 $2\pi$ ごとに複製が並ぶ。

In [ ]:
nuRange, M = 3 * np.pi, 481
nu = np.linspace(-nuRange, nuRange, M)
NUh, NUv = np.meshgrid(nu, nu)                  # 横：nu_h，縦：nu_v
th = np.deg2rad(12)                             # 傾き（垂直軸から）
sgh, sgv = 0.42, 1.05                           # 細長いガウス形（水平に狭く，垂直に広い）


def uspec(h, v):
    return np.exp(-((h * np.cos(th) - v * np.sin(th)) ** 2 / (2 * sgh**2)
                    + (h * np.sin(th) + v * np.cos(th)) ** 2 / (2 * sgv**2)))


U = uspec(NUh, NUv)
Xs = np.zeros_like(U)
for mv in range(-2, 3):
    for mh in range(-2, 3):
        Xs += uspec(NUh - 2 * np.pi * mh, NUv - 2 * np.pi * mv)   # 2*pi*L^{-T}*m（L = I）

cmapS = LinearSegmentedColormap.from_list("white_cool", [(1, 1, 1), ccool])   # 白→青
fig, axs = plt.subplots(1, 2, figsize=(9, 4.6), layout="constrained")
for ax, Z, ttl, box in zip(axs, (U, Xs), (r"$\tilde{u}(\boldsymbol{\nu})$", r"$\tilde{x}(\boldsymbol{\nu})$"),
                           (False, True)):
    ax.imshow(Z, extent=(-nuRange, nuRange, -nuRange, nuRange), origin="lower", cmap=cmapS, vmin=0, vmax=1)
    ax.axvline(0, color=cgray, lw=0.5)          # 原点を通る補助線
    ax.axhline(0, color=cgray, lw=0.5)
    if box:
        ax.plot(np.pi * np.array([-1, 1, 1, -1, -1]), np.pi * np.array([-1, -1, 1, 1, -1]), "--",
                color="k", lw=0.8)              # 基本周期の範囲
    ax.set_xticks([-2 * np.pi, 0, 2 * np.pi],
                  [r"$-2\pi/\Delta_\mathrm{h}$", "$0$", r"$2\pi/\Delta_\mathrm{h}$"], fontsize=12)
    ax.set_yticks([-2 * np.pi, 0, 2 * np.pi],
                  [r"$-2\pi/\Delta_\mathrm{v}$", "$0$", r"$2\pi/\Delta_\mathrm{v}$"], fontsize=12)
    ax.set_xlabel(r"$\nu_\mathrm{h}$", fontsize=14)
    ax.set_ylabel(r"$\nu_\mathrm{v}$", fontsize=14, rotation=0, labelpad=10)
    ax.set_title(ttl, fontsize=14)
plt.show()

## エッジ画像のスペクトル
斜めの直線（エッジ）をもつ画像の振幅スペクトルは，直線と直交する方向に伸びる。画像ごとにさまざまな形のスペクトルをもつ。

In [ ]:
N = 128
rr, cc = np.mgrid[0:N, 0:N]                     # 行番号，列番号（0 始まり）
c0 = N / 2 - 1                                  # 中心
E = ((np.abs((rr - c0) + 0.5 * (cc - c0)) < 1.5) & (np.hypot(rr - c0, cc - c0) < N / 2.5)).astype(float)   # 傾き -1/2 の線分
E = gaussian(E, sigma=0.7)                      # 縁の階段状の折り返しを抑える
S = np.log(1 + np.abs(np.fft.fftshift(np.fft.fft2(E))))
fig, axs = plt.subplots(1, 2, figsize=(7, 3.7), layout="constrained")
show(axs[0], E, "エッジ画像")
show(axs[1], S / S.max(), "振幅スペクトル（対数）")
plt.show()

## 大域的定速移動モデル：時空間画像とスペクトル
一次元の静止信号 $x_0(q)$ が垂直方向に速度 $v$ で動く：$x(q_\mathrm{v},t)=x_0(q_\mathrm{v}-vt)$。縦軸を垂直位置 $q_\mathrm{v}$，横軸を時刻 $t$ とする配列を作り，二次元 DFT の振幅スペクトルを見る（後の標本化の図と同じく，横軸 $\nu_\mathrm{t}$，縦軸 $\nu_\mathrm{v}$）。スペクトルは直線 $\nu_\mathrm{t}=-v\,\nu_\mathrm{v}$ 上に集中し，静止なら $\nu_\mathrm{t}=0$（縦の直線），速いほど傾く。

非整数の速度では $N$ フレームで周期がつながらず，時間方向の端の不連続がスペクトルの漏れ（縦縞）になるので，時間方向にハン窓を掛けてから DFT する。

In [ ]:
x0 = np.exp(-(np.arange(N) - N / 4) ** 2 / (2 * 3**2))   # ガウス形の塊（垂直位置．v=0.5 で折り返さないよう下寄り）
k = np.fft.fftfreq(N, 1 / N)                    # DFT の周波数番号 0,1,...,N/2-1,-N/2,...,-1
wt = signal.windows.hann(N, sym=True)           # 時間方向の窓
vs = [0, 0.5, 1]
tt = np.arange(N)
Xts, Sps = [], []
for v in vs:
    # 行：垂直位置 q_v，列：時刻 t．周波数領域でずらす（非整数の速度も扱える）
    Xt = np.real(np.fft.ifft(np.fft.fft(x0)[:, None] * np.exp(-1j * 2 * np.pi * k[:, None] * v * tt[None, :] / N), axis=0))
    Sp = np.log(1 + np.abs(np.fft.fftshift(np.fft.fft2(Xt * wt[None, :]))))
    Xts.append(Xt)
    Sps.append(Sp / Sp.max())

上段が時空間画像，下段がその振幅スペクトル（左から $v=0,\ 0.5,\ 1$）。時空間画像の軸は左下の角から，スペクトルの軸は原点（中央）を通る矢印で示す。

In [ ]:
def axis_arrows(ax, center, labels, color):
    """軸の向きを矢印で示す（center=True なら中央を通す）．座標は軸の割合で指定する．"""
    o = 0.5 if center else 0.0
    kw = dict(xycoords="axes fraction", textcoords="axes fraction", annotation_clip=False,
              arrowprops=dict(arrowstyle="-|>", color=color, lw=1, mutation_scale=10))
    ax.annotate("", xy=(1.12, o), xytext=(0, o), **kw)
    ax.annotate("", xy=(o, 1.12), xytext=(o, 0), **kw)
    ax.text(1.14, o, labels[0], transform=ax.transAxes, fontsize=14, va="center", ha="left")
    ax.text(o + 0.03, 1.12, labels[1], transform=ax.transAxes, fontsize=14, va="center", ha="left")


fig, axs = plt.subplots(2, 3, figsize=(9, 6.4))
fig.subplots_adjust(left=0.03, right=0.88, bottom=0.03, top=0.9, wspace=0.45, hspace=0.35)
for i, v in enumerate(vs):
    for j, (Z, lab, ctr, ca) in enumerate(((Xts[i], ("$t$", r"$q_\mathrm{v}$"), False, "k"),
                                           (Sps[i], (r"$\nu_\mathrm{t}$", r"$\nu_\mathrm{v}$"), True, cgray))):
        ax = axs[j, i]
        ax.imshow(Z, cmap="gray", origin="lower", interpolation="nearest")
        ax.set_axis_off()
        axis_arrows(ax, ctr, lab, ca)
    axs[0, i].set_title(f"$v={v:g}$", fontsize=13, pad=22)
plt.show()

## 大域的定速移動モデル：三次元のスペクトル（垂直移動の例）
速度 $\boldsymbol{v}=(v_\mathrm{v}\ v_\mathrm{h})^\top$ で動く画像のスペクトルは

$$\tilde{x}(\nu_\mathrm{v},\nu_\mathrm{h},\nu_\mathrm{t})=\tilde{x}_0(\nu_\mathrm{v},\nu_\mathrm{h})\cdot 2\pi\,\delta(\nu_\mathrm{v}v_\mathrm{v}+\nu_\mathrm{h}v_\mathrm{h}+\nu_\mathrm{t})$$

となり，法線ベクトル $(v_\mathrm{v}\ v_\mathrm{h}\ 1)^\top$ と直交する平面の上に集まる。静止画像 $x_0$ のスペクトルが $(\nu_\mathrm{v},\nu_\mathrm{h})=(\pm1,\pm2)$ のインパルスからなるとき，垂直方向に速度 $v_\mathrm{v}$ で動かすと，各インパルスは $\nu_\mathrm{t}$ 方向に $\nu_\mathrm{t}=-v_\mathrm{v}\nu_\mathrm{v}$ の位置へ移る（$v_\mathrm{h}=0$ なので $\nu_\mathrm{h}$ には依らない）。

図の見せ方は，前回の演習・試験用の図（3 次元の動画像スペクトル）を参考にした。軸は $\nu_\mathrm{v}$ を上，$\nu_\mathrm{t}$ を右手前，$\nu_\mathrm{h}$ を右奥にとる。青の点がスペクトル，緑の面がスペクトルの乗る平面，橙の矢印が法線ベクトルである。

In [ ]:
nuv0 = np.array([1, 1, -1, -1])                 # 静止画像のインパルスの垂直周波数 ν_v
nuh0 = np.array([2, -2, 2, -2])                 # 同じく水平周波数 ν_h
vh = 0                                          # 水平速度（垂直移動の例なので 0）
nuts = {}
for vv in [0, 2]:                               # 垂直速度：静止と v_v = 2
    nut = -(nuv0 * vv + nuh0 * vh)              # 移った先の時間周波数 ν_t
    nuts[vv] = nut
    print(f"v_v = {vv}")
    print(f"{'nu_v':>6}{'nu_h':>6}{'nu_t':>6}")
    for row in zip(nuv0, nuh0, nut):
        print("".join(f"{int(e):6d}" for e in row))

三次元の周波数空間にスペクトルの点，平面，法線ベクトルを描く関数。描画座標は $(x,y,z)=(\nu_\mathrm{t},\nu_\mathrm{h},\nu_\mathrm{v})$ とし，$\nu_\mathrm{v}$ を上，$\nu_\mathrm{t}$ を右手前，$\nu_\mathrm{h}$ を右奥に向ける。左が静止，右が $v_\mathrm{v}=2$ で，2 枚は同じ縮尺で描く。

In [ ]:
def drawspec3d(ax, nuv, nuh, nut, vv, vh):
    """三次元の周波数空間にスペクトルの点，平面，法線ベクトルを描く"""
    L = [3.4, 3.4, 2.8]                         # 各軸（ν_t, ν_h, ν_v）の半分の長さ
    # 平面 ν_v v_v + ν_h v_h + ν_t = 0（スペクトルが乗る面）
    pv = np.array([-1.6, -1.6, 1.6, 1.6])
    ph = np.array([-2.8, 2.8, 2.8, -2.8])
    pt = -(pv * vv + ph * vh)
    ax.add_collection3d(Poly3DCollection([list(zip(pt, ph, pv))], facecolor=(*cmain, 0.12),
                                         edgecolor=cmain, linestyle=":"))
    # 座標軸（矢印）と目盛
    axlab = [r"$\nu_\mathrm{t}$", r"$\nu_\mathrm{h}$", r"$\nu_\mathrm{v}$"]
    tk = [[-2, 2], [-2, 2], [-1, 1]]            # 目盛の位置
    tkoff = [[0, 0.6, -0.45], [0, 0, 0.3], [-0.35, 0, 0]]   # 目盛の数字のずらし方
    for a in range(3):
        e = np.zeros(3)
        e[a] = 1
        ax.plot(*[L[a] * np.array([-1, 1]) * e[i] for i in range(3)], color="k", lw=0.8)
        ax.quiver(*(0.8 * L[a] * e), *(0.2 * L[a] * e), color="k", lw=0.8, arrow_length_ratio=0.4)
        p = (L[a] + 0.45) * e
        if a == 1:
            p[2] = -0.45                        # ν_h の名前は軸の下に置く
        ax.text(*p, axlab[a], fontsize=16, ha="center", va="center")
        for t in tk[a]:
            q = t * e
            d = 0.1 * np.array([1, 0, 0]) if a == 2 else 0.1 * np.array([0, 0, 1])
            ax.plot(*[q[i] + np.array([-1, 1]) * d[i] for i in range(3)], color="k", lw=0.8)
            o = tkoff[a]
            if a == 1:
                continue                        # ν_h の目盛の数字は省く（混み合うため）
            if a == 0 and vv == 0:
                continue                        # 静止のときは ν_t の目盛の数字も省く（点と重なるため）
            ax.text(q[0] + o[0], q[1] + o[1], q[2] + o[2], f"{t}", fontsize=12, ha="center", va="center",
                    color=(0.3, 0.3, 0.3))
    # スペクトルの点（インパルス）と，それらを結ぶ補助線
    ordr = [0, 1, 3, 2, 0]
    ax.plot(nut[ordr], nuh[ordr], nuv[ordr], "--", color=cgray)
    ax.scatter(nut, nuh, nuv, s=50, color=ccool, depthshade=False)
    # 法線ベクトル (v_v v_h 1)^T（成分の並びは (ν_v, ν_h, ν_t)）
    ax.quiver(0, 0, 0, 1, vh, vv, color=cwarm, lw=3, arrow_length_ratio=0.25)
    # 数字が点や軸に重ならない位置に置く（静止のときは左上）
    if vv == 0:
        o, ha = [-1.2, 0, 2.1], "right"
    else:
        o, ha = [0.3, 0, 0.4], "left"
    ax.text(1 + o[0], vh + o[1], vv + o[2], rf"$({vv}\ {vh}\ 1)^\top$", fontsize=14, color=cwarm, ha=ha)
    lim = [l + 0.6 for l in L]
    ax.set(xlim=(-lim[0], lim[0]), ylim=(-lim[1], lim[1]), zlim=(-lim[2], lim[2]))
    ax.set_box_aspect(lim, zoom=1.35)           # 縦横の縮尺を揃える
    ax.set_axis_off()
    ax.view_init(elev=25, azim=-50)             # 方位角 -50°，仰角 25° から見る


fig = plt.figure(figsize=(12, 4.6))
for i, vv in enumerate([0, 2]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    drawspec3d(ax, nuv0, nuh0, nuts[vv], vv, vh)
    ax.set_title(f"$v_\\mathrm{{v}}={vv}$", fontsize=14)
fig.subplots_adjust(left=0, right=1, bottom=0, top=0.95, wspace=0)
plt.show()

スライドの例（$v_\mathrm{v}=2$）の数値。$\nu_\mathrm{v}=1$ の成分は $\nu_\mathrm{t}=-2$ に，$\nu_\mathrm{v}=-1$ の成分は $\nu_\mathrm{t}=2$ に移り，いずれも平面の式 $\nu_\mathrm{v}v_\mathrm{v}+\nu_\mathrm{t}=0$ を満たす。

In [ ]:
vv = 2
nutp = -vv * 1                                  # ν_v = 1 の成分の移り先
nutm = -vv * (-1)                               # ν_v = -1 の成分の移り先
print("nutp =", nutp, "，nutm =", nutm)
assert 1 * vv + nutp == 0 and (-1) * vv + nutm == 0
print(f"1・{vv}+({nutp})={1 * vv + nutp}")
# スライドの値と一致することを確かめる
assert (vv, nutp, nutm) == (2, -2, 2)

## ブロックマッチング：前回スライドの例題
$2\times2$ の対象ブロック（現フレーム $\mathsf{x}_{n_\mathrm{t}}$）に最も近いブロックを，参照フレーム $\mathsf{x}_{m_\mathrm{t}}$ の $4\times4$ の探索窓から，差の絶対値和（SAD）を評価式とする全探索で求める。探索範囲は $\mathcal{N}_\mathrm{w}=\{-1,0,1\}^2$ である。

探索窓の中央の $2\times2$（`win[1:3, 1:3]`）が，対象ブロックと同じ位置（$\boldsymbol{m}=\mathbf{0}$）である。変位 $\boldsymbol{m}=[m_\mathrm{v}\ \ m_\mathrm{h}]^\top$ の候補は `win[1+mv:3+mv, 1+mh:3+mh]` で，教科書の $x_{m_\mathrm{t}}[\boldsymbol{n}+\boldsymbol{m}]$ に当たる。

In [ ]:
blk = np.array([[1, 1], [1, 1]])                # 対象ブロック（現フレーム）
win = np.array([[0, 0, 0, 0], [2, 2, 1, 1], [2, 2, 1, 1], [0, 0, 0, 0]])   # 探索窓（参照フレーム）
mvs = [-1, 0, 1]                                # 垂直の変位 m_v の候補
mhs = [-1, 0, 1]                                # 水平の変位 m_h の候補
SAD = np.zeros((len(mvs), len(mhs)), dtype=int) # 行：m_v，列：m_h
SSD = np.zeros_like(SAD)
for iv, mv in enumerate(mvs):
    for ih, mh in enumerate(mhs):
        cand = win[1 + mv:3 + mv, 1 + mh:3 + mh]   # 候補ブロック
        SAD[iv, ih] = np.sum(np.abs(blk - cand))
        SSD[iv, ih] = np.sum((blk - cand) ** 2)
print("SAD =\n", SAD)
print("SSD =\n", SSD)
sadmin = SAD.min()
iv, ih = np.unravel_index(np.argmin(SAD), SAD.shape)
mhat = [mvs[iv], mhs[ih]]                       # 推定した動きベクトル [m_v m_h]（垂直，水平）
print("sadmin =", sadmin, "，mhat =", mhat)

演習課題の解説と同じく，一つの変位について SAD を差の絶対値の和に展開して書いておく。ここでは $\boldsymbol{m}=\mathbf{0}$（動きなし）の場合を示す。

In [ ]:
cand0 = win[1:3, 1:3]
terms = [f"|{p}-{q}|" for p, q in zip(blk.ravel(), cand0.ravel())]
sadexpr = "+".join(terms) + f"={np.sum(np.abs(blk - cand0))}"
print(sadexpr)

スライドの表（SAD と SSD），最小値，推定した動きベクトル，展開式と一致することを確かめる。

In [ ]:
assert np.array_equal(SAD, [[4, 3, 2], [4, 2, 0], [4, 3, 2]])
assert np.array_equal(SSD, [[4, 3, 2], [4, 2, 0], [4, 3, 2]])
assert sadmin == 0 and mhat == [0, 1]
assert sadexpr == "|1-2|+|1-1|+|1-2|+|1-1|=2"

## 動き推定の例（ブロックサイズ 16×16）
静止した背景（参考資料 6 章 COLUMN「一次自己回帰過程」の AR(1) 過程 $x[n]=\rho x[n-1]+w[n]$ を垂直・水平の両方向に適用して白色ノイズから生成した $256\times256$ の画像。$\rho=0.95$ は参考資料が自然画像の値として挙げるもの。物体が目立つよう濃淡を $[0.25,0.7]$ に収める）の上を，模様のある正方形（msipimg07「モンブラン」の中央を切り出して $48\times48$ に縮小したもの，細い線状の模様がある）と明るい円が 1 フレームあたり $[2\ \ 2]^\top$ 画素（垂直，水平とも +2，右下向き）動く 2 フレームを作る。現フレーム $\mathsf{x}_{n_\mathrm{t}}$ を $16\times16$ のブロックに分け，前フレーム $\mathsf{x}_{m_\mathrm{t}}$（$m_\mathrm{t}=n_\mathrm{t}-1$）の $\pm7$ 画素の範囲を全探索する。

教科書の SAD の定義では $\boldsymbol{m}$ は参照フレーム内で一致する位置へのずれなので，物体が $[2\ \ 2]^\top$ 動いたブロックでは $\hat{\boldsymbol{m}}_b=[-2\ \ -2]^\top$（物体の移動と逆向き）が得られる。

In [ ]:
M = 256
rng = np.random.default_rng(0)
rho, burn = 0.95, 64                            # AR(1) の相関係数と，立ち上がりを捨てる画素数
W = rng.standard_normal((M + burn, M + burn))   # 白色ノイズ w
A1 = signal.lfilter([1], [1, -rho], signal.lfilter([1], [1, -rho], W, axis=0), axis=1)   # 垂直・水平に x[n] = rho x[n-1] + w[n]
A1 = A1[burn:, burn:]
bg = 0.25 + 0.45 * (A1 - A1.min()) / (A1.max() - A1.min())   # 背景（静止）：AR(1) 過程の生成画像，濃淡を狭める
Xm = img_as_float(vie.msipimg(7, 512, "gray"))  # モンブラン
tex = resize(Xm[216:296, 216:296], (48, 48), order=3, anti_aliasing=True)   # 正方形の模様（中央 80×80 を 3 次補間で縮小）
Y1, X1 = np.mgrid[0:M, 0:M]                     # 行番号，列番号


def insertobj(bg, tex, psq, pcir):
    """背景 bg に模様 tex の正方形（左上 psq = [行 列]）と明るい円（中心 pcir = [行 列]）を重ねる"""
    F = bg.copy()
    h, w = tex.shape
    F[psq[0]:psq[0] + h, psq[1]:psq[1] + w] = tex
    F[(X1 - pcir[1]) ** 2 + (Y1 - pcir[0]) ** 2 <= 12**2] = 1
    return F


dmov = np.array([2, 2])                         # 物体の 1 フレームあたりの移動 [垂直 水平]


def frame(s):
    return insertobj(bg, tex, np.array([59, 109]) + s, np.array([169, 109]) + s)


F0 = frame(np.array([0, 0]))                    # 参照フレーム（前フレーム）
F1 = frame(dmov)                                # 現フレーム
bs = 16                                         # ブロックサイズ
R = 7                                           # 探索範囲 ±R 画素
nb = M // bs
MV = np.zeros((nb, nb), dtype=int)              # 各ブロックの動きベクトルの垂直・水平成分
MH = np.zeros((nb, nb), dtype=int)
for bi in range(nb):
    for bj in range(nb):
        r0, c0 = bi * bs, bj * bs
        cur = F1[r0:r0 + bs, c0:c0 + bs]        # 現フレームのブロック
        best = np.sum(np.abs(cur - F0[r0:r0 + bs, c0:c0 + bs]))   # m = 0 から始める
        for mv in range(-R, R + 1):
            for mh in range(-R, R + 1):
                rr2, cc2 = r0 + mv, c0 + mh     # 参照フレームの候補ブロックの左上（n + m）
                if rr2 < 0 or cc2 < 0 or rr2 + bs > M or cc2 + bs > M:
                    continue
                sad = np.sum(np.abs(cur - F0[rr2:rr2 + bs, cc2:cc2 + bs]))
                if sad < best - 1e-9:
                    best, MV[bi, bj], MH[bi, bj] = sad, mv, mh
nsearch = (2 * R + 1) ** 2                      # ブロック毎の評価回数
print(f"nsearch = {2 * R + 1}^2 = {nsearch}")

左は現フレーム，右は動きベクトル場である。動きベクトル場は白地に描く（ブロックの境界を灰色の格子で示す）。矢印は $\hat{\boldsymbol{m}}_b$ の向きで，見やすいように 4 倍の長さで描く。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5.2), layout="constrained")
show(axs[0], F1, "現フレーム")
ax = axs[1]
show(ax, np.ones((M, M)), "動きベクトル場")
for g in range(0, M + 1, bs):
    ax.plot([g - 0.5, g - 0.5], [-0.5, M - 0.5], color=(0.85, 0.85, 0.85), lw=0.8)
    ax.plot([-0.5, M - 0.5], [g - 0.5, g - 0.5], color=(0.85, 0.85, 0.85), lw=0.8)
gx, gy = np.meshgrid(np.arange(nb) * bs + bs / 2 - 0.5, np.arange(nb) * bs + bs / 2 - 0.5)
ax.quiver(gx, gy, 4 * MH, 4 * MV, angles="xy", scale_units="xy", scale=1, color=cmain, width=0.008)
ax.plot(gx.ravel(), gy.ravel(), ".", color=cmain, ms=4)
ax.set(xlim=(-0.5, M - 0.5), ylim=(M - 0.5, -0.5))
plt.show()

動いた領域で最も多いベクトルを求める。

In [ ]:
mask = (MV != 0) | (MH != 0)
moving = np.column_stack([MV[mask], MH[mask]])
mode_vec = stats.mode(moving, axis=0, keepdims=False).mode   # 動いた領域で最も多いベクトル [m_v m_h]
print("mode_vec =", mode_vec, "，dmov =", dmov)
print(f"bs = {bs}，R = {R}，count = {2 * R + 1}^2={nsearch}")

## 位相限定相関（POC）
参考資料のサンプル画像 msipimg05（石像の顔，グレースケール，$256\times256$ に縮小）を $\boldsymbol{n}_0=[-3\ \ 5]^\top$ 画素（垂直，水平）ずらした画像 $y[\boldsymbol{n}]=x[\boldsymbol{n}-\boldsymbol{n}_0]$ との POC を計算する。テンプレート $\mathsf{v}$ を元の画像，$\mathsf{x}$ をずらした画像とすると，クロススペクトル $\overline{V}X$ を振幅で正規化して逆変換した POC 配列は，ラグ $\boldsymbol{\ell}=\boldsymbol{n}_0$ に鋭いピークをもつ。

In [ ]:
rng = np.random.default_rng(1)
Xc = img_as_float(vie.msipimg(5, 256, "gray"))  # 石像の顔
n0 = np.array([-3, 5])                          # 与えるずれ [垂直 水平]
Xs = np.roll(Xc, n0, axis=(0, 1)) + 0.01 * rng.standard_normal(Xc.shape)   # y[n] = x[n - n0]（巡回）に少し雑音を加える
Rc = np.conj(np.fft.fft2(Xc)) * np.fft.fft2(Xs)  # クロススペクトル conj(V) X
P = np.real(np.fft.ifft2(Rc / np.maximum(np.abs(Rc), np.finfo(float).eps)))   # POC 配列
pr, pc = np.unravel_index(np.argmax(P), P.shape)
peak = P[pr, pc]
lag = np.mod(np.array([pr, pc]) + 128, 256) - 128   # ピークのラグ [垂直 水平]
print("lag =", lag)
print(f"peak = {peak:.4f}")
print("lag は n0 と一致:", np.array_equal(lag, n0))

原点付近（ラグ $\pm20$）を立体表示する。

In [ ]:
Pd = np.fft.fftshift(P)                         # ラグ 0 を中央へ
lr = np.arange(-20, 21)
LH, LV = np.meshgrid(lr, lr)
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection="3d")
ax.plot_wireframe(LH, LV, Pd[np.ix_(lr + 128, lr + 128)], color=cmain, lw=0.6)
ax.view_init(elev=40, azim=-120)                # 方位角 -120°，仰角 40° から見る
ax.set(xlim=(-20, 20), ylim=(-20, 20))
ax.set_xlabel(r"$\ell_\mathrm{h}$", fontsize=13)
ax.set_ylabel(r"$\ell_\mathrm{v}$", fontsize=13)
ax.set_zlabel(r"$p_{vx}[\ell_\mathrm{v},\ell_\mathrm{h}]$", fontsize=13)
plt.show()

## 見かけ上の動き：光源変化の影響（前回スライドの図を踏襲）
動かない球を，フレーム $n_\mathrm{t}-1$ では左上から，フレーム $n_\mathrm{t}$ では右上から照らす。拡散反射（ランバート反射）の明るさ $a+k\max(0,\boldsymbol{\eta}^\top\boldsymbol{\ell})$（$\boldsymbol{\eta}$：面の法線，$\boldsymbol{\ell}$：光源の向き）で陰影をつけ，床には光と反対側に影を落とす。物体は動いていないのに，明るい面と影が入れ替わり，動いたように見える。

In [ ]:
Ms = 200
xs, ys = np.meshgrid(np.linspace(-1.3, 1.3, Ms), np.linspace(1.3, -1.3, Ms))   # 横：右が正，縦：上が正（縦横同じ縮尺）
r0, yc = 0.75, 0.1                              # 球の半径と中心の高さ
dx, dy = xs / r0, (ys - yc) / r0
inside = dx**2 + dy**2 <= 1
nz = np.sqrt(np.maximum(0, 1 - dx**2 - dy**2))  # 球面の法線の奥行き成分
lights = [np.array([-1, 1, 0.9]), np.array([1, 1, 0.9])]   # 左上から，右上から
fig, axs = plt.subplots(1, 2, figsize=(7, 3.7), layout="constrained")
for ax, lv, ttl in zip(axs, lights, ("フレーム $n_\\mathrm{t}-1$", "フレーム $n_\\mathrm{t}$")):
    l = lv / np.linalg.norm(lv)
    I = 0.93 * np.ones((Ms, Ms))                # 背景（明るい灰）
    ysh = -0.78
    floorY = ys < -0.6                          # 床
    I[floorY] = 0.82
    # 床の影：光と反対側にずれた楕円
    shx = -0.5 * np.sign(l[0])
    sh = ((xs - shx) / 0.85) ** 2 + ((ys - ysh) / 0.14) ** 2 <= 1
    I[sh & floorY] = 0.55
    shade = 0.15 + 0.8 * np.maximum(0, dx * l[0] + dy * l[1] + nz * l[2])
    I[inside] = shade[inside]
    show(ax, I, ttl)
plt.show()

## まとめ
- 動いている映像のスペクトルは速度に応じて傾いた平面に集中する（大域的定速移動モデル）
- 動き推定には，ブロックマッチング法（SAD や SSD の最小化）や位相限定相関が使われる
- 動きベクトルは $\boldsymbol{m}=[m_\mathrm{v}\ \ m_\mathrm{h}]^\top$（垂直，水平）の順に表す
- 見かけ上の動きには，空間勾配の欠落，光源変化，アパーチャ問題，オクルージョンなどの本質的な問題がある

© Copyright, Shogo MURAMATSU, All rights reserved.